In [52]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
from optimizer import OneBitOptimizer
import gc

In [53]:
wine = fetch_openml(name='wine-quality-white', version=1, as_frame=False)
X, y = wine.data, wine.target.astype(int)
y = y - y.min()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_dataset  = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=30, shuffle=False)

In [54]:
classes, counts = np.unique(y, return_counts=True)
total_samples = len(y)
print("Class distribution")
for cls, count in zip(classes, counts):
    percentage = (count / total_samples) * 100
    print(f"Class {cls}: {percentage:.2f}%")

Class distribution
Class 0: 0.41%
Class 1: 3.33%
Class 2: 29.75%
Class 3: 44.88%
Class 4: 17.97%
Class 5: 3.57%
Class 6: 0.10%


In [55]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(11, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 7)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = OneBitOptimizer(model.parameters(), lr=0.01)
gc.collect()

2156

In [56]:
epochs = 10
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [57]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()

Epoch [1/10]
Training loss 138598730572482.72
Training accuracy 8.269525267993874 %
Testing loss 1.9450486630809551
Testing accuracy 19.591836734693878 %
Epoch [2/10]
Training loss 1.9452423720167509
Training accuracy 16.947422154160286 %
Testing loss 1.9450290774812504
Testing accuracy 19.591836734693878 %
Epoch [3/10]
Training loss 1.9452018401280784
Training accuracy 17.559979581419093 %
Testing loss 1.9449628822657528
Testing accuracy 19.591836734693878 %
Epoch [4/10]
Training loss 1.9451686350772306
Training accuracy 7.912200102092904 %
Testing loss 1.944971936089652
Testing accuracy 2.5510204081632653 %
Epoch [5/10]
Training loss 1.945165867462275
Training accuracy 3.5222052067381315 %
Testing loss 1.9449657007139556
Testing accuracy 2.5510204081632653 %
Epoch [6/10]
Training loss 1.945195272724625
Training accuracy 3.5222052067381315 %
Testing loss 1.9450946194784982
Testing accuracy 2.5510204081632653 %
Epoch [7/10]
Training loss 1.945327025764508
Training accuracy 3.5222052067

In [58]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(11, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 7)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = OneBitOptimizer(model.parameters(), lr=0.01)
gc.collect()

8

In [59]:
epochs = 10
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [60]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()

Epoch [1/10]
Training loss 238327687021614.53
Training accuracy 3.5222052067381315 %
Testing loss 1.9508697877124863
Testing accuracy 2.5510204081632653 %
Epoch [2/10]
Training loss 1.9507854324387555
Training accuracy 3.5222052067381315 %
Testing loss 1.9508642724582128
Testing accuracy 2.5510204081632653 %
Epoch [3/10]
Training loss 1.9507535751405085
Training accuracy 3.5222052067381315 %
Testing loss 1.9507390771593367
Testing accuracy 2.5510204081632653 %
Epoch [4/10]
Training loss 1.9506463230477722
Training accuracy 3.5222052067381315 %
Testing loss 1.9507132732138341
Testing accuracy 2.5510204081632653 %
Epoch [5/10]
Training loss 1.95059590926275
Training accuracy 3.5222052067381315 %
Testing loss 1.9505908902810545
Testing accuracy 2.5510204081632653 %
Epoch [6/10]
Training loss 1.9504922680127006
Training accuracy 3.5222052067381315 %
Testing loss 1.9505759088360533
Testing accuracy 2.5510204081632653 %
Epoch [7/10]
Training loss 1.950408486336576
Training accuracy 3.5222052